# H2O AutoML Full-Mode Experiment (Kaggle Kernel)

This notebook runs H2O AutoML in full mode on both data sources (standard, deeppavlov) for OOS detection.

**Prerequisites:**
1. Attach a Kaggle Dataset containing `standard/full.json` and `deeppavlov/full.json`
2. Replace `<DATASET_NAME>` placeholder below with your actual dataset slug (e.g., `username/clinc150-oos-data`)

**Output:**
- `metrics_h2o_full.json` in `/kaggle/working/` (downloadable from kernel output)

## 1. Configuration

**USER ACTION REQUIRED:** Replace `<DATASET_NAME>` with your Kaggle Dataset slug.

In [ ]:
# ============================================================
# USER CONFIGURATION - REPLACE THIS VALUE
# ============================================================
# Replace <DATASET_NAME> with your Kaggle dataset slug
# Example: "username/clinc150-oos-processed"
KAGGLE_DATASET_NAME = "<DATASET_NAME>"
# ============================================================

# Repository settings
REPO_URL = "https://github.com/va-av-8/autoguardrails.git"
REPO_BRANCH = "oos_detection"
REPO_DIR = "/kaggle/working/autoguardrails"

# Data paths
KAGGLE_INPUT_DIR = f"/kaggle/input/{KAGGLE_DATASET_NAME}"
SOURCES = ["standard", "deeppavlov"]

# Validate configuration
if KAGGLE_DATASET_NAME == "<DATASET_NAME>":
    raise ValueError(
        "Please replace <DATASET_NAME> with your actual Kaggle dataset slug!\n"
        "Example: KAGGLE_DATASET_NAME = 'username/clinc150-oos-processed'"
    )

## 2. Environment Setup

Install Java (required by H2O) and Python dependencies.

In [ ]:
%%bash
# Install Java (OpenJDK) - required by H2O
apt-get update -qq
apt-get install -y -qq openjdk-11-jdk-headless > /dev/null 2>&1
echo "Java installed: $(java -version 2>&1 | head -1)"

In [ ]:
# Install Python dependencies
!pip install -q h2o sentence-transformers scikit-learn numpy pandas

In [ ]:
# Verify installations
import subprocess
import sys

# Check Java
java_version = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java:")
print(java_version.stderr.split('\n')[0])

# Check Python packages
import h2o
import sentence_transformers
print(f"\nh2o version: {h2o.__version__}")
print(f"sentence-transformers version: {sentence_transformers.__version__}")

## 3. Clone Repository

In [ ]:
import os
import shutil
from pathlib import Path

# Clone the repository
if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)

!git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}

print(f"\nRepository cloned to: {REPO_DIR}")
print(f"Branch: {REPO_BRANCH}")

In [ ]:
# Add repository to Python path
import sys
sys.path.insert(0, REPO_DIR)

# Verify import works
from tasks.oos_detection.src.experiment_runner import run_framework_grid
from tasks.oos_detection.src.data_utils import load_split
print("Imports successful!")

## 4. Setup Data

Copy data files from Kaggle Dataset to the repository's expected location.

In [ ]:
from pathlib import Path
import shutil

# Define paths
repo_data_dir = Path(REPO_DIR) / "tasks" / "oos_detection" / "data" / "processed"

# Verify input data exists
input_dir = Path(KAGGLE_INPUT_DIR)
if not input_dir.exists():
    raise FileNotFoundError(
        f"Kaggle input directory not found: {input_dir}\n"
        f"Make sure you attached the dataset '{KAGGLE_DATASET_NAME}' to this kernel."
    )

print(f"Input directory contents:")
!ls -la {KAGGLE_INPUT_DIR}

# Copy data for each source
for source in SOURCES:
    src_file = input_dir / source / "full.json"
    if not src_file.exists():
        # Try flat structure (full.json might be at root level with source prefix)
        src_file = input_dir / f"{source}_full.json"
        if not src_file.exists():
            raise FileNotFoundError(
                f"Data file not found for source '{source}'.\n"
                f"Expected: {input_dir / source / 'full.json'} or {input_dir / f'{source}_full.json'}"
            )
    
    # Create destination directory
    dest_dir = repo_data_dir / source
    dest_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy the file
    dest_file = dest_dir / "full.json"
    shutil.copy2(src_file, dest_file)
    print(f"Copied: {src_file} -> {dest_file}")

print("\nData setup complete!")

In [ ]:
# Verify data is loadable
for source in SOURCES:
    train = load_split(source, "train")
    val = load_split(source, "validation")
    test = load_split(source, "test")
    print(f"{source}: train={len(train['texts'])}, val={len(val['texts'])}, test={len(test['texts'])}")

## 5. Run H2O Full-Mode Experiments

This runs H2O AutoML on full training data for both sources.

In [ ]:
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

# Define results path
results_file = Path(REPO_DIR) / "tasks" / "oos_detection" / "results" / "metrics.json"
results_file.parent.mkdir(parents=True, exist_ok=True)

print(f"Results will be saved to: {results_file}")

In [ ]:
%%time

# Run H2O full-mode experiments
results, errors = run_framework_grid(
    frameworks=["h2o"],
    sources=SOURCES,
    run_full=True,
    run_fewshot=False,
    n_shots_list=[],
    seeds=[],
    results_file=str(results_file),
    continue_on_error=True,
)

print(f"\nCompleted experiments: {len(results)}")
if errors:
    print(f"Failed experiments: {len(errors)}")
    for err in errors:
        print(f"  ERROR: {err}")

## 6. Display and Save Results

In [ ]:
import json

# Load results
with open(results_file, "r") as f:
    all_results = json.load(f)

# Filter to H2O full records
h2o_full_results = [
    r for r in all_results 
    if r.get("extra", {}).get("framework") == "h2o" and r.get("mode") == "full"
]

print(f"H2O full-mode records: {len(h2o_full_results)}")
print("=" * 80)

In [ ]:
# Display results in a table
import pandas as pd

if h2o_full_results:
    df = pd.DataFrame([
        {
            "source": r["extra"].get("source", "?"),
            "mode": r["mode"],
            "oos_recall": r["oos_recall"],
            "in_domain_acc": r["in_domain_acc"],
            "f1_oos": r["f1_oos"],
            "auroc": r["auroc"],
            "au_ioc": r["au_ioc"],
            "latency_ms": r["latency_ms"],
        }
        for r in h2o_full_results
    ])
    display(df)
else:
    print("No H2O full results found!")

In [ ]:
# Print full JSON for each result
print("\nFull JSON records:")
print("=" * 80)
for r in h2o_full_results:
    print(json.dumps(r, indent=2))
    print("-" * 40)

In [ ]:
# Save results to Kaggle output directory
output_file = Path("/kaggle/working/metrics_h2o_full.json")

with open(output_file, "w") as f:
    json.dump(h2o_full_results, f, indent=2)

print(f"Results saved to: {output_file}")
print(f"File size: {output_file.stat().st_size} bytes")
print("\nThis file is available in the kernel output for download.")

## 7. Cleanup

Shutdown H2O cluster to free resources.

In [ ]:
try:
    import h2o
    h2o.cluster().shutdown(prompt=False)
    print("H2O cluster shut down.")
except Exception as e:
    print(f"H2O shutdown note: {e}")

---

## Summary

**Output file:** `/kaggle/working/metrics_h2o_full.json`

Download this file from the kernel output and merge into your local `metrics.json`:

```python
import json

# Load local metrics
with open("tasks/oos_detection/results/metrics.json") as f:
    local = json.load(f)

# Load Kaggle results
with open("metrics_h2o_full.json") as f:
    kaggle = json.load(f)

# Merge (Kaggle results will be added)
local.extend(kaggle)

# Save
with open("tasks/oos_detection/results/metrics.json", "w") as f:
    json.dump(local, f, indent=2)
```